In [ ]:
!pip install nltk
!pip install spacy
!pip install pyarrow
!pip install --upgrade pandas pyarrow

In [ ]:
import pandas as pd
import spacy
from collections import Counter
import os

# 1. Setup paths for both files
splits_dir = os.path.join('..', 'data', 'splits')
files_to_process = {
    'train': os.path.join(splits_dir, 'train.parquet'),#manhattan
    'test': os.path.join(splits_dir, 'test.parquet')#philadelphia
}

# 2. Setup spaCy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def extract_complex_landmarks(text_list):
    counts = Counter()
    blacklist = {
        'avenue', 'block', 'turn', 'way', 'left', 'right', 
        'st', 'ave', 'meters', 'feet', 'side', 'corner', 'end', 'direction'
    }

    print(f"Processing {len(text_list)} instructions...")
    # We enable the 'attribute_ruler' and 'lemmatizer' for better noun phrase detection
    for doc in nlp.pipe(text_list.astype(str), batch_size=500, disable=["ner"]):
        
        # --- Strategy: Use noun_chunks to find multi-word landmarks ---
        # This catches "Dunkin Donuts", "Central Park", "Small Cafe", etc.
        for chunk in doc.noun_chunks:
            # Clean the chunk text
            clean_words = [t.text.lower() for t in chunk if t.text.lower() not in blacklist and not t.is_stop and len(t.text) > 2]
            
            if clean_words:
                phrase = " ".join(clean_words)
                counts.update([phrase])

    return counts

# --- EXECUTION ---
all_results = []

for split_name, path in files_to_process.items():
    if os.path.exists(path):
        print(f"✅ Loading {split_name} data...")
        df = pd.read_parquet(path, engine='pyarrow')
        
        # Extract landmarks (including multi-word phrases)
        landmark_counts = extract_complex_landmarks(df['content'])
        
        # Convert to list for DataFrame
        for landmark, freq in landmark_counts.most_common(100):
            all_results.append({
                'Split': split_name,
                'Landmark': landmark.upper(),
                'Frequency': freq
            })
    else:
        print(f"❌ File not found: {path}")

# 3. Save to File
if all_results:
    results_df = pd.DataFrame(all_results)
    output_file = 'frequent_landmarks_report.csv'
    results_df.to_csv(output_file, index=False)
    print(f"\n📁 SUCCESS: Frequent landmarks saved to '{output_file}'")

    # Display Top Multi-word results for the Train set
    print("\n--- Top Multi-Word Landmarks Found ---")
    print(results_df[results_df['Landmark'].str.contains(' ')].head(20))

✅ Loading train data...
Processing 7000 instructions...
✅ Loading test data...
Processing 1278 instructions...

📁 SUCCESS: Frequent landmarks saved to 'frequent_landmarks_report.csv'

--- Top Multi-Word Landmarks Found ---
    Split              Landmark  Frequency
5   train       BICYCLE PARKING       1131
34  train  FAST FOOD RESTAURANT        193
37  train        BICYCLE RENTAL        166
40  train           POST OFFICE        158
42  train              POST BOX        155
49  train  DUANE READE PHARMACY        134
50  train        DRINKING WATER        132
57  train            CHASE BANK        115
62  train          CLOTHES SHOP        112
66  train           PARKING LOT        107
67  train     HISTORIC MEMORIAL        102
72  train          BIKE PARKING         94
75  train     HISTORIC DISTRICT         88
76  train          ALCOHOL SHOP         86
77  train            SOUTH EAST         85
78  train           VACANT SHOP         85
79  train     HISTORIC BUILDING         84
80 